<a href="https://colab.research.google.com/github/ihsann2410010447/statistika-probabilitas-uniska/blob/main/M12_Praktikum_Uji_Satu_Sampel_Taxi_2410010447_Akbar_Ihsan_Nugraha.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚕 Modul 12 — Praktikum Uji Hipotesis Satu Sampel

### Studi Kasus: NYC Taxi (6.433 perjalanan) + Klaim Akurasi Model ML

**Mata Kuliah:** Statistika & Probabilitas (TIF1501)
**Program Studi:** Teknik Informatika (55201) · FTI · Uniska MAB Banjarmasin
**Semester:** 5 · 3 SKS · CPMK072 (Sub-CPMK 072.2) · Milestone 4

---

## 🎯 Tujuan Pembelajaran

1. **Menentukan** arah uji (two-/one-tailed) dan mengambil keputusan via nilai kritis, p-value, dan CI.
2. **Menjalankan** z-test dan t-test satu sampel untuk rata-rata.
3. **Menjalankan** uji proporsi satu sampel (z-test & binomial exact), termasuk klaim akurasi model ML.
4. **Mengecek** asumsi dan memakai uji non-parametrik (sign test, Wilcoxon) serta power analysis.

---

## 📋 Alur Notebook

| Bagian | Isi | Waktu |
|--------|-----|-------|
| **A** | Setup & Load NYC Taxi | 5 menit |
| **B** | EDA Klaim | 10 menit |
| **C** | z-test Durasi: two- vs one-tailed | 15 menit |
| **D** | t-test Tarif & Tip + Dualitas CI | 15 menit |
| **E** | Uji Proporsi: Kartu Kredit & Akurasi Model | 15 menit |
| **F** | Asumsi & Uji Non-parametrik | 15 menit |
| **G** | Power & Ukuran Sampel | 5 menit |
| **H** | Template Milestone 4 + Refleksi | 10 menit |

**Total: ± 90 menit**

---

## 💡 Lima Klaim yang Diuji

| # | Klaim | Uji |
|---|---|---|
| 1 | "Rata-rata durasi perjalanan 15 menit" (σ historis 12 menit) | z-test |
| 2 | "Rata-rata tarif $13" | t-test two-tailed |
| 3 | "Penumpang kartu kredit memberi tip lebih dari 20%" | t-test right-tailed |
| 4 | "70% pembayaran memakai kartu kredit" | z-test proporsi |
| 5 | "Median tarif $10" | sign test |
| Bonus | "Akurasi model ML lebih dari 95%" | binomial exact |

Konteksnya taksi New York, tetapi pola analisisnya sama persis untuk data ojek/taksi online di Indonesia.


---

# 🔧 A. Setup & Load Dataset


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.descriptivestats import sign_test
from statsmodels.stats.power import TTestPower

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
ALPHA = 0.05

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/taxis.csv"
df = pd.read_csv(url, parse_dates=['pickup', 'dropoff'])

# Fitur turunan
df['duration'] = (df['dropoff'] - df['pickup']).dt.total_seconds() / 60   # menit
cc = df[(df['payment'] == 'credit card') & (df['fare'] > 0)].copy()
cc['tip_pct'] = cc['tip'] / cc['fare'] * 100                               # tip (%) penumpang kartu kredit

print(f"Perjalanan        : {len(df):,}")
print(f"Periode           : {df.pickup.min():%d %b %Y} – {df.pickup.max():%d %b %Y}")
print(f"Kartu kredit      : {len(cc):,} perjalanan")
df[['pickup', 'distance', 'fare', 'tip', 'duration', 'color', 'payment']].head()


---

# 🔍 B. EDA Klaim

Lihat dulu bentuk distribusinya — ini menentukan uji mana yang cocok nanti.


In [ ]:
print(df[['fare', 'duration', 'distance']].describe().round(2))
print(f"\nSkewness tarif   : {stats.skew(df.fare):.2f}")
print(f"Skewness durasi  : {stats.skew(df.duration):.2f}")
print(f"Skewness tip %   : {stats.skew(cc.tip_pct):.2f}")

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, data, title, ref, color in [
    (axes[0], df.fare, 'Tarif ($) — klaim mean $13 & median $10', [13, 10], '#1D4ED8'),
    (axes[1], df.duration, 'Durasi (menit) — klaim 15', [15], '#CA8A04'),
    (axes[2], cc.tip_pct, 'Tip % (kartu kredit) — klaim > 20', [20], '#059669')]:
    ax.hist(data, bins=60, color=color, edgecolor='black', alpha=0.75)
    for r in ref:
        ax.axvline(r, color='#E11D48', ls='--', lw=2)
    ax.set_title(title, fontweight='bold')
axes[0].set_xlim(0, 60); axes[1].set_xlim(0, 70)
plt.tight_layout(); plt.show()
print("💡 Tarif & durasi sangat skewed ke kanan. Dengan n ribuan, CLT membuat t-test untuk MEAN tetap aman,")
print("   tetapi untuk klaim MEDIAN atau sampel kecil kita perlu hati-hati (Bagian F).")


## B.1 Fungsi Laporan Uji

Agar setiap uji dilaporkan dengan format yang sama: **statistik, p-value, CI, effect size, keputusan**.


In [ ]:
def laporan(judul, H0, H1, stat_name, stat, p, ci=None, effect=None, alpha=ALPHA):
    print("=" * 62)
    print(judul)
    print("=" * 62)
    print(f"  H₀ : {H0}")
    print(f"  H₁ : {H1}")
    print(f"  {stat_name:<4}= {stat:.4f}")
    print(f"  p  = {p:.4g}")
    if ci is not None:
        print(f"  CI = [{ci[0]:.4f}, {ci[1]:.4f}]")
    if effect is not None:
        print(f"  Effect size: {effect}")
    keputusan = "TOLAK H₀" if p < alpha else "GAGAL TOLAK H₀"
    print(f"  Keputusan (α = {alpha}): {keputusan}\n")
    return keputusan


---

# 📏 C. z-test Durasi: Two-tailed vs One-tailed

**Klaim 1**: "Rata-rata durasi perjalanan 15 menit." Data historis operator menyebut **σ = 12 menit** (σ dianggap diketahui → z-test).

Kita ambil **sampel acak 40 perjalanan** (seperti audit manual).

$$z = \frac{\bar{x} - \mu_0}{\sigma / \sqrt{n}}$$


In [ ]:
MU0, SIGMA, N = 15, 12, 40
sample = df.sample(N, random_state=12)['duration']
xbar = sample.mean()
se = SIGMA / np.sqrt(N)
z = (xbar - MU0) / se

p_two = 2 * stats.norm.sf(abs(z))
p_less = stats.norm.cdf(z)
z_crit_two, z_crit_one = stats.norm.ppf(1 - ALPHA/2), stats.norm.ppf(1 - ALPHA)
ci = (xbar - z_crit_two * se, xbar + z_crit_two * se)

print(f"x̄ = {xbar:.2f} menit, SE = σ/√n = {se:.3f}, z = {z:.3f}\n")
laporan("Z-TEST TWO-TAILED", "μ = 15", "μ ≠ 15", "z", z, p_two, ci)
laporan("Z-TEST LEFT-TAILED", "μ ≥ 15", "μ < 15", "z", z, p_less)

print(f"Pendekatan nilai kritis:")
print(f"  two-tailed : tolak jika |z| > {z_crit_two:.3f}  → |{z:.2f}| {'>' if abs(z) > z_crit_two else '<'} {z_crit_two:.2f}")
print(f"  left-tailed: tolak jika z < −{z_crit_one:.3f} → {z:.2f} {'<' if z < -z_crit_one else '>'} −{z_crit_one:.2f}")
print(f"\nRata-rata SELURUH {len(df):,} perjalanan = {df.duration.mean():.2f} menit")


In [ ]:
# Visualisasi daerah tolak
xs = np.linspace(-4, 4, 400)
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), sharey=True)
for ax, mode, p, title in [(axes[0], 'two', p_two, 'Two-tailed (H₁: μ ≠ 15)'),
                           (axes[1], 'left', p_less, 'Left-tailed (H₁: μ < 15)')]:
    ax.plot(xs, stats.norm.pdf(xs), color='#1E293B', lw=2)
    if mode == 'two':
        for side in [xs < -z_crit_two, xs > z_crit_two]:
            ax.fill_between(xs[side], stats.norm.pdf(xs[side]), color='#E11D48', alpha=0.5)
    else:
        side = xs < -z_crit_one
        ax.fill_between(xs[side], stats.norm.pdf(xs[side]), color='#E11D48', alpha=0.5)
    ax.axvline(z, color='#1D4ED8', lw=3, label=f'z = {z:.2f}')
    ax.set_title(f'{title} → p = {p:.3f}', fontweight='bold'); ax.legend()
plt.tight_layout(); plt.show()
print("⚠️ Data yang SAMA, kesimpulan BERBEDA hanya karena arah uji.")
print("   Arah uji harus ditetapkan dari pertanyaan bisnis SEBELUM data dilihat — bukan setelah tahu x̄ < 15.")


### C.1 Bagaimana jika σ tidak diketahui?

Di praktik, σ populasi hampir tidak pernah diketahui. Bandingkan dengan t-test yang memakai **s sampel**:


In [ ]:
t_stat, p_t = stats.ttest_1samp(sample, MU0)
print(f"s sampel = {sample.std(ddof=1):.2f} menit  (σ historis = {SIGMA})")
print(f"t-test : t = {t_stat:.3f}, p = {p_t:.4f}")
print(f"z-test : z = {z:.3f}, p = {p_two:.4f}")
print("\n💡 Hasil berbeda jauh karena s sampel (≈7) jauh lebih kecil dari σ historis (12).")
print("   Pertanyaan kritis: apakah 'σ historis = 12' masih berlaku? Jika asumsi σ salah, z-test menyesatkan.")
print("   Inilah alasan t-test menjadi pilihan default.")


---

# 🧮 D. t-test Tarif & Tip + Dualitas CI

## D.1 Klaim 2: "Rata-rata tarif $13" (two-tailed)


In [ ]:
fare = df['fare']
t_stat, p = stats.ttest_1samp(fare, 13)
ci_fare = stats.t.interval(1 - ALPHA, len(fare) - 1, loc=fare.mean(), scale=stats.sem(fare))
d = (fare.mean() - 13) / fare.std()
laporan("T-TEST: rata-rata tarif = $13?", "μ = 13", "μ ≠ 13", "t", t_stat, p, ci_fare, f"Cohen's d = {d:.3f}")
print(f"x̄ = ${fare.mean():.2f}, n = {len(fare):,}")
print("💡 'Gagal tolak' bukan berarti terbukti tepat $13 — hanya tidak ada bukti yang menentang klaim.")


## D.2 Klaim 3: "Tip penumpang kartu kredit lebih dari 20%" (right-tailed)

Arah ditetapkan dari klaim: kita hanya ingin tahu apakah tip **melebihi** 20%.


In [ ]:
tp = cc['tip_pct']
t_stat, p = stats.ttest_1samp(tp, 20, alternative='greater')
lower = tp.mean() - stats.t.ppf(1 - ALPHA, len(tp) - 1) * stats.sem(tp)   # batas bawah CI satu arah
d = (tp.mean() - 20) / tp.std()
laporan("T-TEST: tip % > 20?", "μ ≤ 20", "μ > 20", "t", t_stat, p, (lower, np.inf), f"Cohen's d = {d:.3f}")
print(f"x̄ = {tp.mean():.2f}%, s = {tp.std():.2f}%, n = {len(tp):,}")
print(f"Batas bawah CI 95% satu arah = {lower:.2f}% → seluruh interval di atas 20%")


## D.3 Dualitas CI ↔ Uji Hipotesis

Uji two-tailed α = 0.05 menolak H₀ **tepat ketika** μ₀ berada di luar CI 95%. Kita uji beberapa nilai klaim tarif:


In [ ]:
claims = [12.5, 12.8, 13.0, 13.3, 13.5]
rows = []
for mu0 in claims:
    p = stats.ttest_1samp(fare, mu0).pvalue
    inside = ci_fare[0] <= mu0 <= ci_fare[1]
    rows.append([mu0, round(p, 5), 'di dalam' if inside else 'DI LUAR', 'gagal tolak' if p >= ALPHA else 'TOLAK'])
print(f"CI 95% tarif = [{ci_fare[0]:.3f}, {ci_fare[1]:.3f}]\n")
print(pd.DataFrame(rows, columns=['μ₀ klaim', 'p-value', 'posisi vs CI', 'keputusan']).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 2.5))
ax.plot(ci_fare, [0, 0], color='#1D4ED8', lw=8, label='CI 95%')
ax.plot(fare.mean(), 0, 'ko', ms=9, label='x̄')
for mu0, r in zip(claims, rows):
    ax.plot(mu0, 0.3, 'v', ms=12, color='#059669' if r[3] == 'gagal tolak' else '#E11D48')
    ax.text(mu0, 0.45, f'{mu0}', ha='center')
ax.set_ylim(-0.4, 0.8); ax.set_yticks([]); ax.set_xlabel('tarif ($)')
ax.set_title('Hijau = μ₀ di dalam CI (gagal tolak), merah = di luar CI (tolak)', fontweight='bold')
ax.legend(loc='lower right'); plt.tight_layout(); plt.show()


---

# 📊 E. Uji Proporsi Satu Sampel

## E.1 Klaim 4: "70% pembayaran memakai kartu kredit"

$$z = \frac{\hat{p} - p_0}{\sqrt{p_0(1-p_0)/n}}$$


In [ ]:
pay = df.dropna(subset=['payment'])
k, n = (pay['payment'] == 'credit card').sum(), len(pay)
p0 = 0.70
print(f"Syarat: n·p₀ = {n*p0:.0f} ≥ 10, n·(1−p₀) = {n*(1-p0):.0f} ≥ 10 ✅\n")

z, p = proportions_ztest(k, n, value=p0, prop_var=p0)   # SE memakai p₀
ci = proportion_confint(k, n, alpha=ALPHA, method='wilson')
laporan("Z-TEST PROPORSI: kartu kredit = 70%?", "p = 0.70", "p ≠ 0.70", "z", z, p, ci,
        f"selisih = {(k/n - p0)*100:+.2f} poin persen")

exact = stats.binomtest(k, n, p0)
print(f"Binomial exact test  : p = {exact.pvalue:.4f}  (hampir sama dengan z-test karena n besar)")
print(f"\np̂ = {k:,}/{n:,} = {k/n:.2%}")
print("💡 Signifikan secara statistik, tetapi selisihnya hanya ~1.6 poin.")
print("   Apakah 1.6 poin mengubah keputusan bisnis? Itu pertanyaan praktis, bukan statistik.")


## E.2 Bonus ML: "Akurasi model lebih dari 95%"

Setiap prediksi di data uji adalah percobaan Bernoulli (benar/salah) → jumlah benar ~ Binomial. Kita latih Logistic Regression pada dataset kanker payudara (bawaan scikit-learn, tidak perlu unduh).


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)

benar = int((model.predict(X_te) == y_te).sum())
n_uji = len(y_te)
res = stats.binomtest(benar, n_uji, p=0.95, alternative='greater')
ci_acc = proportion_confint(benar, n_uji, alpha=ALPHA, method='beta')   # Clopper-Pearson (exact)

laporan("BINOMIAL EXACT: akurasi > 95%?", "p ≤ 0.95", "p > 0.95", "acc", benar / n_uji, res.pvalue, ci_acc)
print(f"Benar {benar} dari {n_uji} data uji → akurasi {benar/n_uji:.2%}")


In [ ]:
# Seberapa besar data uji yang dibutuhkan untuk membuktikan > 95%?
print(f"{'n data uji':>10} | {'akurasi 100%':>14} | {'akurasi 98%':>14}")
print("-" * 46)
for n_test in [20, 30, 59, 100, 200, 500]:
    p_perfect = stats.binomtest(n_test, n_test, 0.95, alternative='greater').pvalue
    k98 = int(round(0.98 * n_test))
    p98 = stats.binomtest(k98, n_test, 0.95, alternative='greater').pvalue
    fmt = lambda v: f"p = {v:.4f}" + (" ✅" if v < ALPHA else " ❌")
    print(f"{n_test:>10} | {fmt(p_perfect):>14} | {fmt(p98):>14}")
print("\n💡 Dengan 30 data uji, model yang 100% benar pun BELUM terbukti > 95% (p = 0.95³⁰ ≈ 0.21).")
print("   Minimal 59 data uji yang semuanya benar. Untuk skripsi/projek ML: laporkan n data uji & CI akurasi.")


---

# 🔬 F. Asumsi & Uji Non-parametrik

## F.1 Sampel kecil yang tidak normal: taksi hijau, n = 20

Klaim: "Rata-rata tarif taksi hijau $13". Auditor hanya mengambil 20 perjalanan.


In [ ]:
green = df[df.color == 'green'].sample(20, random_state=3)['fare']
print("Tarif terurut:", sorted(green.round(2).tolist()))
print(f"Mean = ${green.mean():.2f}, Median = ${green.median():.2f}")
w, p_sw = stats.shapiro(green)
print(f"Shapiro-Wilk: W = {w:.3f}, p = {p_sw:.2e} → {'normal' if p_sw > ALPHA else 'TIDAK normal'}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(green, bins=15, color='#0D9488', edgecolor='black')
axes[0].set_title('Tarif 20 taksi hijau — satu outlier $150', fontweight='bold')
stats.probplot(green, dist='norm', plot=axes[1])
axes[1].set_title('QQ-plot: titik menjauh dari garis = tidak normal', fontweight='bold')
plt.tight_layout(); plt.show()

t_stat, p_t = stats.ttest_1samp(green, 13)
M, p_sign = sign_test(green, 13)
print(f"t-test (mean = 13)   : p = {p_t:.3f}  ← tidak dapat dipercaya: n kecil & sangat tidak normal")
print(f"sign test (median=13): p = {p_sign:.3f}  ← valid tanpa asumsi bentuk distribusi")
print("💡 Keduanya gagal tolak, tetapi alasannya berbeda: mean $20.97 'terlihat' jauh dari 13 karena satu outlier,")
print("   sementara median $8.75 justru di BAWAH 13. Dengan n = 20, buktinya belum cukup ke arah mana pun.")


## F.2 Klaim 5: "Median tarif $10" — Sign test vs Wilcoxon

- **Sign test**: hitung berapa nilai di bawah vs di atas $10, lalu uji binomial p = 0.5. Tanpa asumsi bentuk.
- **Wilcoxon signed-rank**: memakai *peringkat* besar selisih. Mengasumsikan distribusi **simetris** di sekitar median.


In [ ]:
below, above, equal = (fare < 10).sum(), (fare > 10).sum(), (fare == 10).sum()
print(f"Median sampel = ${fare.median():.2f}")
print(f"Di bawah $10 = {below:,}, di atas = {above:,}, tepat $10 = {equal} (dibuang)\n")

M, p_sign = sign_test(fare, 10)
p_sign_exact = stats.binomtest(below, below + above, 0.5).pvalue
laporan("SIGN TEST: median = $10?", "M = 10", "M ≠ 10", "M", M, p_sign)
print(f"  (setara binomtest({below}, {below+above}, 0.5): p = {p_sign_exact:.2e})\n")

w_two = stats.wilcoxon(fare - 10)
w_less = stats.wilcoxon(fare - 10, alternative='less')
print(f"Wilcoxon two-sided : p = {w_two.pvalue:.2e}")
print(f"Wilcoxon 'less'    : p = {w_less.pvalue:.4f}   ← TIDAK mendukung 'median < 10'!")
print(f"\nSkewness tarif = {stats.skew(fare):.2f} → asumsi simetri Wilcoxon DILANGGAR.")
print("💡 Tarif di atas $10 bisa sangat jauh (hingga $150) sehingga peringkatnya besar dan 'menarik' Wilcoxon")
print("   ke arah sebaliknya. Untuk median data skewed, gunakan sign test.")


---

# ⚡ G. Power & Ukuran Sampel

Berapa perjalanan yang dibutuhkan untuk mendeteksi tip rata-rata 21% vs klaim 20% (one-tailed, power 80%)?


In [ ]:
power = TTestPower()
s_tip = tp.std()
for target in [25, 23, 22, 21, 20.5]:
    d = (target - 20) / s_tip
    n_req = power.solve_power(effect_size=d, alpha=ALPHA, power=0.8, alternative='larger')
    print(f"Tip sebenarnya {target:>5}% (d = {d:.3f}) → n ≈ {np.ceil(n_req):>6,.0f} perjalanan")

ds = np.linspace(0.05, 0.6, 50)
plt.plot(ds, [power.solve_power(effect_size=d, alpha=ALPHA, power=0.8, alternative='larger') for d in ds], lw=2.5, color='#1D4ED8')
plt.yscale('log'); plt.xlabel("Cohen's d"); plt.ylabel('n dibutuhkan (log)')
plt.title('Efek makin kecil → n tumbuh ~ 1/d²', fontweight='bold'); plt.tight_layout(); plt.show()


## G.1 Ringkasan Semua Uji


In [ ]:
ringkas = pd.DataFrame([
    ['Durasi = 15 mnt (two-tailed)', 'z-test', p_two],
    ['Tarif = $13', 't-test', stats.ttest_1samp(fare, 13).pvalue],
    ['Tip % > 20', 't-test right-tailed', stats.ttest_1samp(tp, 20, alternative='greater').pvalue],
    ['Kartu kredit = 70%', 'z-test proporsi', proportions_ztest(k, n, value=0.70, prop_var=0.70)[1]],
    ['Akurasi model > 95%', 'binomial exact', res.pvalue],
    ['Median tarif = $10', 'sign test', p_sign],
], columns=['Klaim', 'Uji', 'p-value'])
ringkas['Keputusan'] = np.where(ringkas['p-value'] < ALPHA, 'Tolak H₀', 'Gagal tolak H₀')
ringkas['p-value'] = ringkas['p-value'].apply(lambda v: f"{v:.2e}" if v < 0.001 else f"{v:.4f}")
ringkas


---

# 📝 H. Template Milestone 4 + Refleksi

## H.1 Template untuk Dataset Projek Anda

Cari **nilai acuan** yang bermakna untuk dataset Anda: angka dari literatur, laporan resmi (BPS, laporan industri), standar, target bisnis, atau klaim yang beredar. Tuliskan sumbernya.


In [ ]:
# =========================================================
# TEMPLATE MILESTONE 4 — ganti dengan dataset & kolom Anda
# =========================================================
# df_my = pd.read_csv("<URL dataset Milestone 1-3 Anda>")

# ---------- UJI 1: MEAN ----------
# Sumber μ₀ : "<kutip sumber nilai acuan>"
# Arah uji  : two-sided / greater / less  ← alasan: ...
# x = df_my['<kolom>'].dropna()
# print(stats.shapiro(x.sample(min(len(x), 500), random_state=0)), stats.skew(x))
# t, p = stats.ttest_1samp(x, MU0, alternative='two-sided')
# ci = stats.t.interval(0.95, len(x)-1, loc=x.mean(), scale=stats.sem(x))
# laporan("...", "μ = ...", "μ ≠ ...", "t", t, p, ci, f"d = {(x.mean()-MU0)/x.std():.3f}")

# ---------- UJI 2: PROPORSI ----------
# k, n = (df_my['<kolom>'] == '<kategori>').sum(), df_my['<kolom>'].notna().sum()
# z, p = proportions_ztest(k, n, value=P0, prop_var=P0)
# laporan("...", "p = ...", "p ≠ ...", "z", z, p, proportion_confint(k, n, method='wilson'))

# ---------- UJI 3: MEDIAN (data skewed) ----------
# M, p = sign_test(df_my['<kolom>'].dropna(), MEDIAN0)

# ---------- KESIMPULAN ----------
# Tulis di cell markdown: hasil statistik + apakah selisihnya penting secara praktis

print("💡 Copy template ini ke notebook projek Anda. Submit sebelum pertemuan Modul 13.")


## H.2 Pertanyaan Refleksi

1. **Arah uji**: Pada Bagian C, data yang sama menghasilkan p = 0.055 (two-tailed) dan p = 0.027 (one-tailed). Kapan one-tailed sah dipakai? Berikan satu contoh klaim di dunia IT yang memang wajar diuji one-tailed.

2. **z vs t**: Mengapa z-test dan t-test pada sampel durasi memberi kesimpulan sangat berbeda? Apa risikonya jika kita memakai "σ historis" yang sudah tidak berlaku?

3. **Signifikan vs penting**: Proporsi kartu kredit 71.6% berbeda signifikan dari 70%. Jelaskan mengapa hasil ini belum tentu berarti apa-apa bagi bisnis, dan informasi apa yang dibutuhkan untuk memutuskannya.

4. **Akurasi model**: Teman Anda melaporkan "model saya 100% akurat" berdasarkan 25 data uji. Bagaimana Anda menanggapinya dengan hasil Bagian E.2?

5. **Asumsi**: Mengapa Wilcoxon memberi kesimpulan yang bertentangan dengan sign test pada klaim median tarif $10? Kapan Wilcoxon tetap aman dipakai?

### ✏️ Jawaban Anda:

**1. Arah Uji (One-tailed vs Two-tailed)**
*   **Kapan sah digunakan:** Uji *one-tailed* (satu arah) hanya sah digunakan jika arah perbedaan atau efek sudah dihipotesiskan secara kuat secara teoretis sebelum data dikumpulkan, dan deviasi ke arah sebaliknya benar-benar tidak memiliki konsekuensi praktis atau tidak mungkin terjadi.
*   **Contoh di dunia IT:** Menguji apakah sistem optimasi kompresi gambar baru menghasilkan ukuran file yang *lebih kecil* dibanding metode lama ($H_1: \mu_{baru} < \mu_{lama}$). Kita tidak peduli atau tidak mengharapkan skenario ukuran file menjadi lebih besar.

**2. Analisis z-test vs t-test**
*   **Penyebab perbedaan kesimpulan:** z-test mengasumsikan standar deviasi populasi ($\sigma$) diketahui secara pasti (dalam kasus ini $\sigma = 12$). Sementara t-test mengestimasi variabilitas langsung dari standar deviasi sampel ($s \approx 7.07$). Karena variabilitas sampel jauh lebih kecil daripada klaim historis, pembagi pada rumus statistik uji mengecil, sehingga nilai $t$ dan $z$ serta $p$-value yang dihasilkan menjadi sangat berbeda.
*   **Risiko memakai $\sigma$ historis yang usang:** Jika variabilitas populasi sesungguhnya telah berubah (misal karena perubahan rute jalan atau kebijakan lalu lintas baru), menggunakan $\sigma$ historis akan memicu kesalahan Tipe I (menolak $H_0$ padahal benar) atau Tipe II (gagal menolak $H_0$ padahal salah), sehingga hasil keputusan statistik menjadi bias dan menyesatkan.

**3. Signifikansi Statistik vs Signifikansi Praktis**
*   **Mengapa belum tentu berarti bagi bisnis:** Dengan ukuran sampel yang sangat besar ($n > 6000$), uji statistik memiliki *power* yang sangat tinggi untuk mendeteksi perbedaan sekecil apa pun (seperti selisih 1.6%). Namun, perbedaan 1.6% ini secara riil mungkin tidak menutupi biaya investasi atau perubahan sistem operasional.
*   **Informasi yang dibutuhkan:** Diperlukan analisis biaya-manfaat (*cost-benefit analysis*). Misalnya, berapa biaya transaksi *merchant* kartu kredit dibandingkan dengan uang tunai, dan apakah peningkatan 1.6% pengguna kartu kredit benar-benar memberikan tambahan profit margin bersih yang sepadan.

**4. Tanggapan Klaim Akurasi Model 100%**
*   **Tanggapan:** Klaim akurasi 100% dari hanya 25 data uji tidak cukup kuat secara statistik untuk membuktikan akurasi model di atas 95% pada populasi umum. Berdasarkan distribusi Binomial (*exact binomial test*), jika $n=25$ dan semuanya benar ($k=25$), nilai $p$-value untuk menguji $H_0: p \le 0.95$ vs $H_1: p > 0.95$ adalah $0.95^{25} \approx 0.277$. Karena $p > 0.05$, kita gagal menolak $H_0$. Kita memerlukan minimal 59 data uji yang semuanya benar untuk membuktikan secara signifikan bahwa akurasi model memang di atas 95%.

**5. Analisis Asumsi Wilcoxon vs Sign Test**
*   **Mengapa bertentangan:** Data tarif taksi memiliki *skewness* kanan yang sangat ekstrem (banyak nilai murah, sedikit nilai yang sangat mahal/outlier hingga $150). Wilcoxon mengasumsikan bahwa distribusi data berbentuk simetris di sekitar median. Keberadaan nilai-nilai besar (outlier) memberikan bobot peringkat (*rank*) yang sangat besar, sehingga menarik hasil uji Wilcoxon ke arah yang salah. Di sisi lain, *Sign Test* hanya peduli pada tanda (+/-) tanpa melihat magnitudo nilai, sehingga robust terhadap *skewness* dan outlier.
*   **Kapan Wilcoxon aman dipakai:** Wilcoxon *signed-rank* aman digunakan ketika data bertipe ordinal atau kontinu yang tidak berdistribusi normal, namun distribusinya masih bersifat **simetris** (memiliki bentuk sisi kiri dan kanan yang relatif seimbang di sekitar median, misal distribusi seragam/uniform).

## H.3 Checklist Submit

### Praktikum (notebook ini)
- [ ] Semua cell berjalan tanpa error
- [ ] z-test durasi two- vs one-tailed + perbandingan dengan t-test
- [ ] t-test tarif & tip + tabel dualitas CI
- [ ] Uji proporsi kartu kredit (z-test & exact)
- [ ] Uji akurasi model + tabel ukuran data uji
- [ ] Shapiro, QQ-plot, sign test vs Wilcoxon
- [ ] Kurva power & tabel ringkasan semua uji
- [ ] 5 pertanyaan refleksi dijawab

### Milestone 4 (notebook projek Anda)
- [ ] Minimal 3 uji satu sampel (mean, proporsi, median)
- [ ] Sumber setiap nilai acuan (μ₀ / p₀ / median₀) dituliskan
- [ ] Arah uji ditetapkan & dijustifikasi sebelum uji dijalankan
- [ ] Cek asumsi + uji alternatif bila asumsi dilanggar
- [ ] p-value + CI + effect size + kesimpulan praktis
- [ ] README diperbarui, commit: `"Milestone 4: one-sample hypothesis tests"`
- [ ] Link GitHub dikumpulkan via LMS **sebelum Modul 13**

---

## 🎓 Selamat! Anda Berhasil...

✅ Menentukan arah uji dan memahami bahaya memilihnya setelah melihat data
✅ Menjalankan z-test, t-test, uji proporsi, dan binomial exact
✅ Menguji klaim akurasi model ML dengan benar
✅ Mengecek asumsi dan memilih sign test untuk data skewed

## 📚 Bacaan Lanjutan
- **Bruce, P.** (2020). *Practical Statistics for Data Scientists*, Ch. 3 (Statistical Experiments & Significance Testing).
- **Wasserstein, R. & Lazar, N.** (2016). *The ASA Statement on p-Values: Context, Process, and Purpose*. The American Statistician.
- **Dokumentasi SciPy** — `scipy.stats.ttest_1samp`, `binomtest`, `wilcoxon`.

---

*Notebook by: Prodi Teknik Informatika · Uniska MAB Banjarmasin · Edisi 2026*
